## shouldSplit: train + inference

Ниже короткий рабочий блок для:
1. обучения `shouldSplit` модели через существующий pipeline,
2. сохранения артефакта,
3. инференса по новым объявлениям из артефакта.

In [1]:
from pathlib import Path
import os
import sys

repo_root = Path("C:/Users/ditriy/Desktop/hack-mfti").resolve()
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("cwd:", Path.cwd())
print("pythonpath has repo root:", str(repo_root) in sys.path)

cwd: C:\Users\ditriy\Desktop\hack-mfti
pythonpath has repo root: True


In [25]:
import avito

In [27]:
from pathlib import Path

import joblib
import pandas as pd

from avito.classifier import train_should_split_models
from avito.config import AvitoCaseConfig
from avito.embeddings import EncoderConfig, SentenceTransformerEncoder
from avito.features import ShouldSplitFeatureConfig
from avito.inference import predict_should_split_from_artifact

case_config = AvitoCaseConfig.from_default_yaml()
feature_config = ShouldSplitFeatureConfig(
    include_extra_text_features=case_config.should_split.include_extra_text_features,
 )

data_path = Path("avito/data/rnc_dataset.csv")
artifact_path = Path("checkpoints/avito_should_split_model.joblib")

train_df = pd.read_csv(data_path)

# Обучаем с эмбеддингами
encoder = SentenceTransformerEncoder(EncoderConfig.from_default_yaml())
result = train_should_split_models(
    df=train_df,
    include_embeddings=True,
    encoder=encoder,
    feature_config=feature_config,
    training_config=case_config.should_split.training,
 )

artifact_path.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "best_model_name": result.model_name,
    "pipeline": result.pipeline,
    "with_embeddings": True,
    "feature_config": feature_config.model_dump(mode="json"),
}
joblib.dump(payload, artifact_path)

sample_df = train_df.iloc[:5][["description", "sourceMcId", "sourceMcTitle"]].copy()
inference = predict_should_split_from_artifact(
    df=sample_df,
    artifact_path=artifact_path,
    encoder=encoder,
 )

sample_df["shouldSplit_pred"] = inference.predictions
sample_df["shouldSplit_proba"] = inference.probabilities
sample_df.head()

c:\Users\ditriy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_name" in TrainedShouldSplitModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\ditriy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_should_split_ratio" in TrainedShouldSplitModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
c:\Users\ditriy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:132: UserWarning: Field "model_comparison_records" in TrainedShouldSplitModel has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.

Loading weights:   0%|          | 0/119 [00:00<?, ?it/s]

Default prompt name is set to 'Classification'. This prompt will be applied to all `encode()` calls, except if `encode()` is called with `prompt` or `prompt_name` parameters.
2026-03-29 02:10:13,216 - avito-embeddings - INFO - [AVITO/EMBEDDINGS] Запуск encode для 3000 текстов


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

2026-03-29 02:10:38,929 - avito-should-split - INFO - [SHOULD_SPLIT] ========================================================================================
2026-03-29 02:10:38,936 - avito-should-split - INFO - [SHOULD_SPLIT] Старт сравнения архитектур shouldSplit
2026-03-29 02:10:38,941 - avito-should-split - INFO - [SHOULD_SPLIT] Кандидатов в сравнении: 6
2026-03-29 02:10:38,947 - avito-should-split - INFO - [SHOULD_SPLIT] ========================================================================================
2026-03-29 02:10:38,953 - avito-should-split - INFO - [SHOULD_SPLIT] ========================================================================================
2026-03-29 02:10:38,957 - avito-should-split - INFO - [SHOULD_SPLIT] [1/6] Архитектура: logistic_regression
2026-03-29 02:10:38,960 - avito-should-split - INFO - [SHOULD_SPLIT] Базовые параметры: {'max_iter': 1200, 'class_weight': 'balanced'}
2026-03-29 02:10:38,966 - avito-should-split - INFO - [SHOULD_SPLIT] Обучение на

...

2026-03-29 02:11:22,104 - avito-should-split - INFO - [SHOULD_SPLIT] Обучение завершено.
2026-03-29 02:11:22,142 - avito-should-split - INFO - [SHOULD_SPLIT] Результат на валидации:
2026-03-29 02:11:22,147 - avito-should-split - INFO - [SHOULD_SPLIT]   gt_should_split_ratio    = 0.3710
2026-03-29 02:11:22,148 - avito-should-split - INFO - [SHOULD_SPLIT]   model_should_split_ratio = 0.4273
2026-03-29 02:11:22,158 - avito-should-split - INFO - [SHOULD_SPLIT]   ratio_delta              = -0.056293
2026-03-29 02:11:22,161 - avito-should-split - INFO - [SHOULD_SPLIT]   ratio_abs_delta          = 0.056293
2026-03-29 02:11:22,161 - avito-should-split - INFO - [SHOULD_SPLIT] ========================================================================================
2026-03-29 02:11:22,169 - avito-should-split - INFO - [SHOULD_SPLIT] [5/6] Архитектура: xgboost
2026-03-29 02:11:22,179 - avito-should-split - INFO - [SHOULD_SPLIT] Базовые параметры: {'n_estimators': 350, 'max_depth': 6, 'learning_rat

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

2026-03-29 02:11:34,905 - avito-should-split - INFO - [SHOULD_SPLIT] Обучение завершено.
2026-03-29 02:11:34,941 - avito-should-split - INFO - [SHOULD_SPLIT] Результат на валидации:
2026-03-29 02:11:34,949 - avito-should-split - INFO - [SHOULD_SPLIT]   gt_should_split_ratio    = 0.3710
2026-03-29 02:11:34,954 - avito-should-split - INFO - [SHOULD_SPLIT]   model_should_split_ratio = 0.4116
2026-03-29 02:11:34,959 - avito-should-split - INFO - [SHOULD_SPLIT]   ratio_delta              = -0.040633
2026-03-29 02:11:34,965 - avito-should-split - INFO - [SHOULD_SPLIT]   ratio_abs_delta          = 0.040633
2026-03-29 02:11:34,971 - avito-should-split - INFO - [SHOULD_SPLIT] Итог базового сравнения:
2026-03-29 02:11:34,974 - avito-should-split - INFO - [SHOULD_SPLIT] Лучшая архитектура до тюнинга: random_forest; ratio_abs_delta=0.004839
2026-03-29 02:11:34,979 - avito-should-split - INFO - [SHOULD_SPLIT] ========================================================================================
2

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,description,sourceMcId,sourceMcTitle,shouldSplit_pred,shouldSplit_proba
0,штукатурка под обои в офис. Делаю ремонт штука...,108,Штукатурные работы,False,0.276405
1,Косметический ремонт под ключ в доме. Отдельно...,101,Ремонт квартир и домов под ключ,True,0.882375
2,напольные покрытия /монтаж перегородок выезд з...,109,Напольные покрытия,False,0.425750
3,Монтаж сантехники в коттедж. Делаю монтаж бойл...,102,Сантехника,False,0.211746
4,"Комплекс работ по ремонту во вторичке. гкл, гк...",101,Ремонт квартир и домов под ключ,False,0.148058
